# 02 — Chunking + retrieval test

Goal: for each clause type, does the right chunk show up in the top-k search results? If recall@5 is low here, the LLM never sees the clause, so fix this before writing any prompts.

In [11]:
import sys, json
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

sys.path.insert(0, "../src")
from chunking import chunk_contract

d = json.load(open("../data/CUADv1.json"))["data"]
split = json.load(open("../data/split.json"))
dev_ids = set(split["dev"])
docs = {x["title"]: x for x in d if x["title"] in dev_ids}
print(len(docs), "dev contracts")

410 dev contracts


## Clause types and search queries
Queries are based on CUAD's own description of each clause type (`category_descriptions.csv` in the CUAD repo).

In [12]:
QUERIES = {
    # Original 10
    "Termination For Convenience": "Either party may terminate this agreement at any time without cause upon written notice.",
    "Post-Termination Services": "Obligations that continue after termination or expiration, such as transition services, final payments, return of materials.",
    "Exclusivity": "Exclusive dealing: a party must buy or sell only through the other party, or is the exclusive distributor or supplier.",
    "Renewal Term": "This agreement shall automatically renew for successive renewal terms unless either party gives notice of non-renewal.",
    "Insurance": "A party shall maintain insurance coverage, such as general liability insurance, with minimum limits and name the other party as additional insured.",
    "Minimum Commitment": "Minimum purchase or order quantity, minimum amount or units that must be bought each year or period.",
    "Ip Ownership Assignment": "Intellectual property, work product and inventions created shall be owned by and assigned to the other party.",
    "Change Of Control": "If a party undergoes a change of control, merger, or sale of its assets, the other party may terminate or must consent.",
    "Non-Compete": "A party shall not compete with the other party or engage in a competing business in a territory or field.",
    "Uncapped Liability": "Limitations of liability shall not apply to breaches of confidentiality, indemnification, gross negligence or willful misconduct.",
    # Added 5
    "Notice Period To Terminate Renewal": "Either party may prevent renewal by giving written notice of non-renewal at least ninety days before the end of the then-current term.",
    "Governing Law": "This agreement shall be governed by and construed in accordance with the laws of the State of New York.",
    "Cap On Liability": "In no event shall either party's aggregate liability under this agreement exceed the total fees paid in the twelve months preceding the claim.",
    "Anti-Assignment": "Neither party may assign or transfer this agreement or its rights without the prior written consent of the other party.",
    "Liquidated Damages": "A party shall pay liquidated damages or a termination fee as a reasonable estimate of loss and not as a penalty.",
}

# Optional keyword boost: a chunk containing the key word gets a small bonus.
# Off by default so scores come from plain embedding search.
USE_KEYWORDS = False
BOOST = 0.15
KEYWORDS = {
    "Non-Compete": r"\bcompet(e|es|ing|ition|itive|itor)",
    "Exclusivity": r"\bexclusiv(e|ely|ity)\b",
    "Minimum Commitment": r"\bminimum\b",
    "Ip Ownership Assignment": r"intellectual property|work product|invention",
    "Notice Period To Terminate Renewal": r"non-?renewal|prior to the (end|expiration)",
    "Governing Law": r"governed by|laws of the state",
    "Cap On Liability": r"aggregate liability|in no event|shall not exceed",
    "Anti-Assignment": r"\bassign",
    "Liquidated Damages": r"liquidated damages|termination fee|penalt",
}
print(len(QUERIES), "clause types")

15 clause types


In [13]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
q_emb = model.encode(list(QUERIES.values()), normalize_embeddings=True)
q_emb = dict(zip(QUERIES.keys(), q_emb))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Gold spans for the 10 clause types

In [14]:
gold = {}  # (contract, clause) -> list of (start, end)
for title, doc in docs.items():
    for qa in doc["paragraphs"][0]["qas"]:
        clause = qa["id"].split("__")[-1]
        if clause in QUERIES and not qa["is_impossible"]:
            gold[(title, clause)] = [(a["answer_start"], a["answer_start"] + len(a["text"])) for a in qa["answers"]]
print(len(gold), "contract-clause pairs with an answer")

2265 contract-clause pairs with an answer


## Retrieve and score
A hit means at least one of the top-k chunks overlaps a gold span. Start with 50 contracts; this cell is the slow one.

In [15]:
import re
K_VALUES = [1, 3, 5, 10]
SAMPLE = sorted(docs)[:50]   # change to sorted(docs) for all 410 dev contracts

def overlaps(c, spans):
    return any(c["start"] < e and s < c["end"] for s, e in spans)

patterns = {k: re.compile(v, re.I) for k, v in KEYWORDS.items()}

rows = []
for title in SAMPLE:
    text = docs[title]["paragraphs"][0]["context"]
    chunks = chunk_contract(text)
    emb = model.encode([c["text"] for c in chunks], normalize_embeddings=True, batch_size=64)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb.astype("float32"))

    for clause, q in q_emb.items():
        spans = gold.get((title, clause))
        if not spans:
            continue
        # score every chunk, then optionally add the keyword boost
        D, I = index.search(q[None].astype("float32"), len(chunks))
        scores = np.zeros(len(chunks))
        scores[I[0]] = D[0]
        if USE_KEYWORDS and clause in patterns:
            scores += BOOST * np.array([bool(patterns[clause].search(c["text"])) for c in chunks])
        ranked = [chunks[i] for i in np.argsort(-scores)]

        row = {"contract": title, "clause": clause}
        for k in K_VALUES:
            row[f"r@{k}"] = any(overlaps(c, spans) for c in ranked[:k])
        rows.append(row)

res = pd.DataFrame(rows)
table = res.groupby("clause")[[f"r@{k}" for k in K_VALUES]].mean().round(2)
table["n"] = res.groupby("clause").size()   # how many examples each score is based on
table.sort_values("r@5")

,r@1,r@3,r@5,r@10,n
clause,,,,,
Non-Compete,0.00,0.14,0.36,0.57,14
Exclusivity,0.20,0.50,0.65,0.70,20
Liquidated Damages,0.14,0.57,0.71,1.00,7
Minimum Commitment,0.29,0.57,0.71,0.81,21
Ip Ownership Assignment,0.43,0.71,0.79,0.93,14
Governing Law,0.47,0.69,0.82,0.96,45
Anti-Assignment,0.60,0.80,0.89,0.97,35
Cap On Liability,0.43,0.79,0.89,0.93,28
Change Of Control,0.67,0.78,0.89,0.89,9


In [16]:
res[[f"r@{k}" for k in K_VALUES]].mean().round(2)

r@1     0.50
r@3     0.72
r@5     0.83
r@10    0.90
dtype: float64

## Look at misses
Pick the worst clause type from the table above and read what retrieval returned instead.

In [17]:
CLAUSE = "Uncapped Liability"   # change to the worst one
miss = res[(res.clause == CLAUSE) & (~res["r@5"])].head(3)
for title in miss.contract:
    text = docs[title]["paragraphs"][0]["context"]
    for s, e in gold[(title, CLAUSE)]:
        print("GOLD:", text[s:e][:400])
        print()

GOLD: In no event will either party be liable to the other for special,       incidental, or indirect damages or for any consequential damages       (including lost profits or savings), even if they are informed of the       possibility; provided that this Section 10 does not apply to Customer's       failure to pay any amounts owing to IBM under this Agreement (including       amounts owing for Service

